## Dependensi

In [ ]:
# Core
import sys
import os
import math

# Numerik & Array
import numpy as np

# Image Processing
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import cv2

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Metrik kualitas gambar
from skimage.metrics import structural_similarity as ssim

In [ ]:
# Membuat folder hasil proses
os.makedirs("output", exist_ok=True)

## Dasar Dasar Pemrosesan Citra Digital

### Bagian 1: Membuka, Membaca, dan Menampilkan Gambar

In [ ]:
# a. Menggunakan Pillow (PIL)
img_pil = Image.open("/content/cover3.png")

print("Informasi Gambar (Pillow):")
print(f"Tipe objek : {type(img_pil)}")
print(f"Format file : {img_pil.format}")
print(f"Mode warna : {img_pil.mode}")
print(f"Ukuran (W x H) : {img_pil.size[0]} x {img_pil.size[1]}")
print(f"Info DPI : {img_pil.info.get('dpi', 'N/A')}")

print()

# Konversi PIL -> Numpy Array
img_pil_array = np.array(img_pil)
print("Sebagai Numpy Array:")
print(f"Shape : {img_pil_array.shape}")
print(f"Dtype : {img_pil_array.dtype}")
print(f"Min/Max : {img_pil_array.min()} / {img_pil_array.max()}")
print(f"Memory : {img_pil_array.nbytes:,} bytes"
      f"({img_pil_array.nbytes/1024:.1f} KB)")

In [ ]:
# b. Menggunakan OpenCV
img_cv = cv2.imread("/content/cover3.png")

print("Informasi gambar (OpenCV)")
print(f"Tipe Objek : {type(img_cv)}")
print(f"Shape : {img_cv.shape}")
print(f"Dtype : {img_cv.dtype}")
print(f"Height : {img_cv.shape[0]} px")
print(f"Width : {img_cv.shape[1]} px")
print(f"Channel : {img_cv.shape[2]}")

print("OpenCV menggunakan urutan BGR (bukan RBG)")
print(f"Pixel [0, 0] OpenCV (BGR) : {img_cv[0, 0]}")
print(f"Pixel [0, 0] Pillow (RGB) : {img_pil_array[0, 0]}")


In [ ]:
# c. Konversi BRG -> RGB
img_cv_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
print(f"Setelah konversi BRG -> RGB : {img_cv_rgb[0, 0]}")

In [ ]:
# d. Menampilkan Gambar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Pillow (langsung RGB)
axes[0].imshow(img_pil)
axes[0].set_title(f"Pillow (RGB) \n Size: {img_pil.size}", fontsize=11)
axes[0].axis('off')

# OpenCV Tanpa konversi (BRG / warna terbalik)
axes[1].imshow(img_cv)
axes[1].set_title(f"OpenCV TANPA Konversi -> warna salah", fontsize=11)
axes[1].axis('off')

# OpenCV dengan Konversi (RGB)
axes[2].imshow(img_cv_rgb)
axes[2].set_title(f"OpenCV DENGAN Konversi -> warna benar", fontsize=11)
axes[2].axis('off')

plt.suptitle("Membuka & Menampilkan Gambar",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("output/buka_gambar.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ----- 1e. Mengakses & Memodifikasi Pixel Individual -----
print("\n Akses Pixel Individual:")

# Baca 1 pixel di posisi (100, 200) → (baris, kolom) = (y, x)
pixel_rgb = img_pil_array[100, 200]
print(f"   Pixel (y=100, x=200) RGB = {pixel_rgb}")
print(f"     Red   = {pixel_rgb[0]}")
print(f"     Green = {pixel_rgb[1]}")
print(f"     Blue  = {pixel_rgb[2]}")

# Modifikasi pixel: buat kotak merah 50×50 pixel
img_modified = img_pil_array.copy()
img_modified[50:100, 50:100] = [255, 0, 0]    # Kotak merah
img_modified[50:100, 110:160] = [0, 255, 0]   # Kotak hijau
img_modified[50:100, 170:220] = [0, 0, 255]   # Kotak biru
img_modified[50:100, 230:280] = [255, 255, 0] # Kotak kuning

plt.figure(figsize=(8, 6))
plt.imshow(img_modified)
plt.title("Modifikasi Pixel: Kotak Warna Ditambahkan", fontsize=12)
plt.axis('off')
plt.savefig("output/B1_pixel_modified.png", dpi=150, bbox_inches='tight')
plt.show()


### Bagian 2: Menyimpan Gambar dalam Berbagai Format & Analisis Ukuran

In [ ]:
img = Image.open("/content/cover3.png")

# a. Menyimpan ke berbagai format
save_configs = {
    # (nama_file, parameter_simpan)
    "JPEG Q=10": ("output/save_jpeg_q10.jpg", {"quality": 10}),
    "JPEG Q=30": ("output/save_jpeg_q30.jpg", {"quality": 30}),
    "JPEG Q=50": ("output/save_jpeg_q50.jpg", {"quality": 50}),
    "JPEG Q=75": ("output/save_jpeg_q75.jpg", {"quality": 75}),
    "JPEG Q=95": ("output/save_jpeg_q95.jpg", {"quality": 95}),
    "JPEG Q=100": ("output/save_jpeg_q100.jpg", {"quality": 100}),
    "PNG": ("output/save_png.png", {}),
    "PNG Compress": ("output/save_png_comp.png", {"compress_level": 9}),
    "BMP": ("output/save_bmp.bmp", {}),
    "TIFF": ("output/save_tiff.tiff", {}),
    "WebP Q=80": ("output/save_webp.webp", {"quality": 80}),
}

results = {}
print(f"Menyimpan gambar dalam berbagai format: \n")
print(f"{'Format':<16} {'Ukuran File':>12} {'Rasio':>8}")
print(f"{'-'*16} {'-'*12} {'-'*8}")

for name, (filepath, params) in save_configs.items():
  img.save(filepath, **params)
  size_bytes = os.path.getsize(filepath)
  results[name] = size_bytes
  ratio = size_bytes / list(results.values())[0] if results else 1.0
  print(f"{name:<16} {size_bytes:>9} B {size_bytes/1024:>6.1f} KB")

In [ ]:
# ----- 2b. Visualisasi Perbandingan Ukuran File -----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart ukuran file
names = list(results.keys())
sizes_kb = [v / 1024 for v in results.values()]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(names)))

bars = axes[0].barh(names, sizes_kb, color=colors, edgecolor='white')
axes[0].set_xlabel("Ukuran File (KB)")
axes[0].set_title("Perbandingan Ukuran File per Format")
axes[0].invert_yaxis()

# Tambah label nilai
for bar, kb in zip(bars, sizes_kb):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 f'{kb:.1f} KB', va='center', fontsize=9)

# Line chart khusus JPEG quality
jpeg_qualities = [10, 30, 50, 75, 95, 100]
jpeg_sizes = [
    results[f"JPEG Q={q}"] / 1024 for q in jpeg_qualities
]

axes[1].plot(jpeg_qualities, jpeg_sizes, 'o-', linewidth=2,
             markersize=8, color='#e74c3c')
axes[1].fill_between(jpeg_qualities, jpeg_sizes, alpha=0.1, color='red')
axes[1].set_xlabel("JPEG Quality")
axes[1].set_ylabel("Ukuran File (KB)")
axes[1].set_title("Pengaruh JPEG Quality terhadap Ukuran File")
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(jpeg_qualities)

for q, s in zip(jpeg_qualities, jpeg_sizes):
    axes[1].annotate(f'{s:.1f} KB', (q, s),
                     textcoords="offset points", xytext=(0, 12),
                     ha='center', fontsize=9)

plt.suptitle("B.2 — Analisis Format File Gambar",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B2_format_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ----- 2c. Perbandingan Visual Kualitas JPEG -----
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
jpeg_files = [
    ("output/save_jpeg_q10.jpg", "JPEG Q=10"),
    ("output/save_jpeg_q30.jpg", "JPEG Q=30"),
    ("output/save_jpeg_q50.jpg", "JPEG Q=50"),
    ("output/save_jpeg_q75.jpg", "JPEG Q=75"),
    ("output/save_jpeg_q95.jpg", "JPEG Q=95"),
    ("output/save_jpeg_q100.jpg", "JPEG Q=100"),
]

for ax, (fpath, label) in zip(axes.flat, jpeg_files):
    img_loaded = Image.open(fpath)
    fsize = os.path.getsize(fpath)
    ax.imshow(img_loaded)
    ax.set_title(f"{label}\n{fsize/1024:.1f} KB", fontsize=11)
    ax.axis('off')

plt.suptitle("Perbandingan Visual Kualitas JPEG",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B2_jpeg_quality_visual.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Baca gambar
img_bgr = cv2.imread("/content/cover3.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# ----- 3a. Konversi ke Berbagai Ruang Warna -----
conversions = {
    'RGB (Original)': img_rgb,
    'Grayscale': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
    'HSV': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV),
    'LAB': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB),
    'YCrCb': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb),
    'HLS': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HLS),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, (name, img_conv) in zip(axes.flat, conversions.items()):
    if len(img_conv.shape) == 2:
        ax.imshow(img_conv, cmap='gray')
    else:
        ax.imshow(img_conv)
    ax.set_title(f"{name}\nShape: {img_conv.shape}", fontsize=11)
    ax.axis('off')

plt.suptitle("B.3a — Konversi ke Berbagai Ruang Warna",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B3_color_spaces.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3b. Visualisasi Channel Terpisah (RGB) -----
R, G, B = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Baris 1: Channel sebagai grayscale
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original RGB", fontsize=12)

axes[0, 1].imshow(R, cmap='gray')
axes[0, 1].set_title(f"Red Channel\nMean={R.mean():.1f}", fontsize=11)

axes[0, 2].imshow(G, cmap='gray')
axes[0, 2].set_title(f"Green Channel\nMean={G.mean():.1f}", fontsize=11)

axes[0, 3].imshow(B, cmap='gray')
axes[0, 3].set_title(f"Blue Channel\nMean={B.mean():.1f}", fontsize=11)

# Baris 2: Channel sebagai warna asli
axes[1, 0].imshow(img_rgb)
axes[1, 0].set_title("Original", fontsize=12)

# Hanya tampilkan channel Red (set G dan B = 0)
red_only = np.zeros_like(img_rgb)
red_only[:,:,0] = R
axes[1, 1].imshow(red_only)
axes[1, 1].set_title("Red Only", fontsize=11)

green_only = np.zeros_like(img_rgb)
green_only[:,:,1] = G
axes[1, 2].imshow(green_only)
axes[1, 2].set_title("Green Only", fontsize=11)

blue_only = np.zeros_like(img_rgb)
blue_only[:,:,2] = B
axes[1, 3].imshow(blue_only)
axes[1, 3].set_title("Blue Only", fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.suptitle("B.3b — Dekomposisi Channel RGB",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B3_rgb_channels.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3c. Visualisasi Channel HSV -----
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
H, S, V = img_hsv[:,:,0], img_hsv[:,:,1], img_hsv[:,:,2]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Original RGB")

axes[1].imshow(H, cmap='hsv')
axes[1].set_title(f"Hue (warna)\nRange: 0–179, Mean={H.mean():.1f}")

axes[2].imshow(S, cmap='gray')
axes[2].set_title(f"Saturation (kemurnian)\nRange: 0–255, Mean={S.mean():.1f}")

axes[3].imshow(V, cmap='gray')
axes[3].set_title(f"Value (kecerahan)\nRange: 0–255, Mean={V.mean():.1f}")

for ax in axes:
    ax.axis('off')

plt.suptitle("B.3c — Dekomposisi Channel HSV",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B3_hsv_channels.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ----- 3d. Konversi RGB→Grayscale Manual vs Built-in -----
print("\n Perbandingan Metode RGB → Grayscale:")

# Metode 1: Average
gray_avg = np.mean(img_rgb, axis=2).astype(np.uint8)

# Metode 2: Luminosity (ITU-R BT.709)
gray_luminosity = (
    0.2126 * img_rgb[:,:,0].astype(float) +
    0.7152 * img_rgb[:,:,1].astype(float) +
    0.0722 * img_rgb[:,:,2].astype(float)
).astype(np.uint8)

# Metode 3: Luminosity (ITU-R BT.601 / NTSC)
gray_ntsc = (
    0.299 * img_rgb[:,:,0].astype(float) +
    0.587 * img_rgb[:,:,1].astype(float) +
    0.114 * img_rgb[:,:,2].astype(float)
).astype(np.uint8)

# Metode 4: OpenCV built-in
gray_cv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Metode 5: Desaturation (max+min)/2
gray_desat = (
    (np.max(img_rgb, axis=2).astype(float) +
     np.min(img_rgb, axis=2).astype(float)) / 2
).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
grays = [
    (img_rgb, "Original RGB", None),
    (gray_avg, "Average\n(R+G+B)/3", 'gray'),
    (gray_luminosity, "Luminosity BT.709\n0.2126R+0.7152G+0.0722B", 'gray'),
    (gray_ntsc, "NTSC BT.601\n0.299R+0.587G+0.114B", 'gray'),
    (gray_cv, "OpenCV cvtColor\n(built-in)", 'gray'),
    (gray_desat, "Desaturation\n(max+min)/2", 'gray'),
]

for ax, (img_g, title, cmap) in zip(axes.flat, grays):
    if cmap:
        ax.imshow(img_g, cmap=cmap)
    else:
        ax.imshow(img_g)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle("B.3d — Perbandingan Metode RGB → Grayscale",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B3_grayscale_methods.png", dpi=150, bbox_inches='tight')
plt.show()

# Hitung perbedaan antar metode
print(f"\n   Perbedaan antar metode (Mean Absolute Error):")
print(f"   Average   vs OpenCV : {np.mean(np.abs(gray_avg.astype(int) - gray_cv.astype(int))):.2f}")
print(f"   BT.709    vs OpenCV : {np.mean(np.abs(gray_luminosity.astype(int) - gray_cv.astype(int))):.2f}")
print(f"   NTSC      vs OpenCV : {np.mean(np.abs(gray_ntsc.astype(int) - gray_cv.astype(int))):.2f}")
print(f"   Desaturat vs OpenCV : {np.mean(np.abs(gray_desat.astype(int) - gray_cv.astype(int))):.2f}")


In [ ]:
# ----- 3e. Konversi Mode Warna Pillow -----
print("\n Konversi Mode Warna Pillow:")
img_pil = Image.open("/content/cover3.png")

modes = {
    'L':     img_pil.convert('L'),      # Grayscale 8-bit
    '1':     img_pil.convert('1'),      # Binary (1-bit)
    'P':     img_pil.convert('P'),      # Palette (8-bit)
    'CMYK':  img_pil.convert('CMYK'),   # Print colors
    'LA':    img_pil.convert('LA'),     # Grayscale + Alpha
}

for mode_name, img_mode in modes.items():
    print(f"   Mode '{mode_name}': size={img_mode.size}, "
          f"bands={img_mode.getbands()}")

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (name, img_m) in zip(axes, modes.items()):
    if name in ['L', '1']:
        ax.imshow(img_m, cmap='gray')
    else:
        ax.imshow(img_m)
    ax.set_title(f"Mode: '{name}'", fontsize=11)
    ax.axis('off')

plt.suptitle("B.3e — Mode Warna Pillow", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("output/B3_pil_modes.png", dpi=150, bbox_inches='tight')
plt.show()